#### Lab Assessment 6
#### Name: Jayesh Kumawat
#### Reg. No.: 26MAG0088
#### Subject: Foundations of Data Science   Course Code: PAAML504


## Aim 
### Implement a MapReduce-based word frequency counter using Python’s multiprocessing module, and benchmark its performance across varying dataset sizes and worker counts.

In [3]:
%%writefile mapreduce_wc.py
import argparse
import multiprocessing as mp
import re
import time
import zlib
from collections import Counter

# Regular expression used to extract words
TOKEN_RE = re.compile(r"[A-Za-z0-9']+")

def read_chunks(file_path, lines_per_chunk):
    chunk = []
    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        for line in file:
            chunk.append(line)
            if len(chunk) == lines_per_chunk:
                yield chunk
                chunk = []
    if chunk:
        yield chunk

def mapper(lines):
    local_counts = Counter()
    for line in lines:
        words = TOKEN_RE.findall(line)
        for word in words:
            local_counts[word.lower()] += 1
    return local_counts

def map_reduce_word_count(file_path, workers=2, reducers=4, lines_per_chunk=10000):
    partitions = [Counter() for _ in range(reducers)]
    with mp.Pool(processes=workers) as pool:
        chunks = read_chunks(file_path, lines_per_chunk)
        for partial_counts in pool.imap_unordered(mapper, chunks, chunksize=1):
            for word, count in partial_counts.items():
                reducer_id = zlib.crc32(word.encode("utf-8")) % reducers
                partitions[reducer_id][word] += count
    final_counts = Counter()
    for partition in partitions:
        final_counts.update(partition)
    return final_counts

def save_counts(counts, output_file):
    sorted_counts = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    with open(output_file, "w", encoding="utf-8") as file:
        for word, count in sorted_counts:
            file.write(f"{word}\t{count}\n")

def main():
    parser = argparse.ArgumentParser(description="MapReduce Word Frequency Counter")
    parser.add_argument("input", help="Input text file")
    parser.add_argument("--output", default="word_counts.txt", help="Output file")
    parser.add_argument("--workers", type=int, default=2, help="Number of mapper processes")
    parser.add_argument("--reducers", type=int, default=4, help="Number of reducer partitions")
    parser.add_argument("--chunk-lines", type=int, default=10000, help="Lines processed by one mapper task")
    parser.add_argument("--top", type=int, default=20, help="Number of top words to display")
    
    args = parser.parse_args()
    start_time = time.perf_counter()
    
    counts = map_reduce_word_count(
        file_path=args.input, workers=args.workers, reducers=args.reducers, lines_per_chunk=args.chunk_lines
    )
    
    elapsed_time = time.perf_counter() - start_time
    save_counts(counts, args.output)
    
    print("MapReduce completed\n-------------------")
    print(f"Total words: {sum(counts.values()):,}\nUnique words: {len(counts):,}")
    print(f"Execution time: {elapsed_time:.3f} seconds\nOutput file: {args.output}")
    print(f"\nTop {args.top} words:\n-------------------")
    for word, count in counts.most_common(args.top):
        print(f"{word:<20} {count}")

if __name__ == "__main__":
    mp.freeze_support()
    main()

Writing mapreduce_wc.py


In [4]:
!python mapreduce_wc.py fintech_reviews.txt --workers 4 --reducers 4 --chunk-lines 10000 --output fintech_top_words.txt --top 20

MapReduce completed
-------------------
Total words: 4,805,483
Unique words: 52,876
Execution time: 0.859 seconds
Output file: fintech_top_words.txt

Top 20 words:
-------------------
good                 213017
app                  210401
to                   120863
very                 115313
not                  113528
is                   111505
the                  111433
and                  88434
i                    86860
it                   73869
this                 65360
nice                 52917
bank                 52677
in                   44439
worst                40025
for                  39936
of                   38308
service              37153
my                   36491
a                    33167


In [9]:
import os
import time
import pandas as pd
import mapreduce_wc

# 1. Create a dedicated folder for our real data slices
os.makedirs("fintech_benchmark_data", exist_ok=True)

def create_fintech_subset(source_file, target_file, num_lines):
    with open(source_file, "r", encoding="utf-8") as src, \
         open(target_file, "w", encoding="utf-8") as tgt:
        for i, line in enumerate(src):
            if i >= num_lines:
                break
            tgt.write(line)

dataset_sizes = [10_000, 50_000, 100_000, 250_000]
worker_values = [1, 2, 4]
records = []
source_txt = "fintech_reviews.txt"

print("Running Benchmark...")

for num_lines in dataset_sizes:
    path = f"fintech_benchmark_data/data_{num_lines}.txt"
    create_fintech_subset(source_txt, path, num_lines)
    size_mb = os.path.getsize(path) / 1_000_000
    
    for workers in worker_values:
        start = time.perf_counter()
        counts = mapreduce_wc.map_reduce_word_count(
            file_path=path, workers=workers, reducers=4, lines_per_chunk=5000
        )
        elapsed = time.perf_counter() - start
        throughput = size_mb / elapsed if elapsed > 0 else 0
        
        records.append({
            "Lines": num_lines,
            "Size_MB": round(size_mb, 2),
            "Workers": workers,
            "Time_seconds": round(elapsed, 3),
            "Throughput_MB_per_second": round(throughput, 2),
            "Unique_words": len(counts),
        })

# THIS is the variable your tables were looking for!
results_df = pd.DataFrame(records) 
print("Benchmark Complete! results_df is now saved in memory.")

Running Benchmark...
Benchmark Complete! results_df is now saved in memory.


In [10]:
# Step 7: Execution Time by Dataset Size and Worker Count
time_table = results_df.pivot(
    index="Size_MB", columns="Workers", values="Time_seconds"
)
time_table.columns = [f"{c}_workers" for c in time_table.columns]
time_table

,1_workers,2_workers,4_workers
Size_MB,,,
0.52,0.271,0.230,0.174
2.20,0.296,0.345,0.267
4.94,0.538,0.375,0.454
11.11,1.053,0.689,0.543


In [11]:
# Step 8: Throughput by Dataset Size and Worker Count
throughput_table = results_df.pivot(
    index="Size_MB", columns="Workers", values="Throughput_MB_per_second"
)
throughput_table.columns = [f"{c}_workers" for c in throughput_table.columns]
throughput_table

,1_workers,2_workers,4_workers
Size_MB,,,
0.52,1.90,2.24,2.97
2.20,7.42,6.36,8.24
4.94,9.17,13.16,10.87
11.11,10.56,16.12,20.48


In [13]:
display(results_df)

,Lines,Size_MB,Workers,Time_seconds,Throughput_MB_per_second,Unique_words
0,10000,0.52,1,0.271,1.90,5206
1,10000,0.52,2,0.230,2.24,5206
2,10000,0.52,4,0.174,2.97,5206
3,50000,2.20,1,0.296,7.42,11650
4,50000,2.20,2,0.345,6.36,11650
5,50000,2.20,4,0.267,8.24,11650
6,100000,4.94,1,0.538,9.17,18058
7,100000,4.94,2,0.375,13.16,18058
8,100000,4.94,4,0.454,10.87,18058
9,250000,11.11,1,1.053,10.56,30332
